# RAGAS Synthetic Test Data Generation from Uploaded PDFs

This notebook generates synthetic RAG test data from user-uploaded PDF files. It includes two generation paths:

- **LangChain documents** loaded with `PyPDFLoader`, split with `RecursiveCharacterTextSplitter`, then passed to RAGAS.
- **LlamaIndex documents** loaded with `SimpleDirectoryReader`, split with `SentenceSplitter`, then passed to RAGAS.

The notebook is generation-only; evaluation examples have been removed.

## 1. Install dependencies

Run once per session. On Colab, restart the runtime after installing before continuing.

In [ ]:
%pip install \
    ragas==0.3.9 \
    langchain==0.3.27 \
    langchain-openai==0.3.28 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.1.2 \
    openai==1.99.5 \
    datasets==4.0.0 \
    pandas==2.2.3 \
    sentence-transformers==3.4.1 \
    rapidfuzz==3.10.1 \
    pypdf==5.9.0 \
    llama-index \
    llama-index-llms-azure-openai \
    llama-index-embeddings-huggingface \
    llama-index-readers-file
print('Install complete. If running on Colab, restart the runtime before proceeding.')

## 2. Imports, configuration, and model setup

Set credentials and all reusable generation settings here only.

In [ ]:
import os
from pathlib import Path
from typing import Iterable

from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import AzureChatOpenAI

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document as LlamaIndexDocument
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.azure_openai import AzureOpenAI as LlamaIndexAzureOpenAI

from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms.base import LangchainLLMWrapper
from ragas.testset import TestsetGenerator

# --- Azure OpenAI credentials and model settings ---
AZURE_OPENAI_API_KEY = ""
AZURE_OPENAI_ENDPOINT = ""
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"
AZURE_OPENAI_DEPLOYMENT = ""  # Your Azure GPT-5 deployment name
AZURE_OPENAI_MODEL_NAME = ""  # Example: "gpt-5"

# --- Local embedding and generation settings ---
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TESTSET_SIZE = 10
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
UPLOAD_DIR = Path("uploaded_pdfs")

# Keep only the columns needed by your downstream evaluation/reporting.
# Update this list after inspecting generated_df.columns for your RAGAS version.
KEEP_COLUMNS = [
    "user_input",
    "reference",
    "reference_contexts",
    "synthesizer_name",
]

os.environ["AZURE_OPENAI_API_KEY"] = AZURE_OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_OPENAI_API_VERSION"] = AZURE_OPENAI_API_VERSION
UPLOAD_DIR.mkdir(exist_ok=True)


# GPT-5 rejects any temperature other than 1. RAGAS otherwise forces a low
# temperature on the LLM (0.01, or 0.3 when n > 1), so pin temperature=1.
class GPT5LangchainLLMWrapper(LangchainLLMWrapper):
    def generate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return super().generate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

    async def agenerate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return await super().agenerate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)


def compact_testset_dataframe(testset, keep_columns: list[str] = KEEP_COLUMNS):
    generated_df = testset.to_pandas()
    available_columns = [column for column in keep_columns if column in generated_df.columns]
    compact_df = generated_df[available_columns].copy()

    print("All generated columns:")
    print(list(generated_df.columns))
    return compact_df


# --- LangChain clients for RAGAS test generation ---
langchain_azure_llm = AzureChatOpenAI(
    openai_api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT.strip(),
    azure_deployment=AZURE_OPENAI_DEPLOYMENT,
    model=AZURE_OPENAI_MODEL_NAME,
    temperature=1,
    validate_base_url=False,
)
langchain_generator_llm = GPT5LangchainLLMWrapper(langchain_azure_llm, bypass_temperature=True)

langchain_hf_embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
)
langchain_generator_embeddings = LangchainEmbeddingsWrapper(langchain_hf_embeddings)

# --- LlamaIndex clients for RAGAS test generation ---
llamaindex_llm = LlamaIndexAzureOpenAI(
    model=AZURE_OPENAI_MODEL_NAME,
    deployment_name=AZURE_OPENAI_DEPLOYMENT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=1,
)
llamaindex_embeddings = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL_NAME)

print("Configuration and model clients initialized.")

## 3. Upload source PDF documents

Upload one or more PDF files. These uploaded PDFs are the only source material used for both generation paths.

In [ ]:
def upload_pdf_files(upload_dir: Path = UPLOAD_DIR) -> list[Path]:
    """Upload any number of PDFs in Colab, or use local PDF_PATHS outside Colab."""
    try:
        from google.colab import files  # type: ignore
    except ImportError:
        pdf_paths = [Path(p) for p in globals().get("PDF_PATHS", [])]
        pdf_paths = [p for p in pdf_paths if p.exists() and p.suffix.lower() == ".pdf"]
        if not pdf_paths:
            raise RuntimeError(
                "This notebook is not running in Colab. Set PDF_PATHS to a list of local PDF files, then rerun this cell."
            )
        return pdf_paths

    uploaded = files.upload()  # Select one or many PDFs in the file picker.
    pdf_paths: list[Path] = []

    for filename, content in uploaded.items():
        if not filename.lower().endswith(".pdf"):
            print(f"Skipping non-PDF file: {filename}")
            continue

        output_path = upload_dir / Path(filename).name
        output_path.write_bytes(content)
        pdf_paths.append(output_path)

    if not pdf_paths:
        raise ValueError("No PDF files were uploaded. Please upload at least one .pdf file.")

    return pdf_paths


pdf_paths = upload_pdf_files()
print(f"Uploaded/selected {len(pdf_paths)} PDF file(s).")
for pdf_path in pdf_paths:
    print(f"- {pdf_path.name}")

## 4. LangChain test data generation

Load the PDFs as LangChain documents, split them, and generate a RAGAS test set.

In [ ]:
def load_langchain_pdf_documents(pdf_paths: Iterable[Path]) -> list[Document]:
    """Load uploaded PDFs into LangChain documents, preserving source metadata."""
    documents: list[Document] = []

    for pdf_path in pdf_paths:
        loader = PyPDFLoader(str(pdf_path))
        pages = loader.load()

        for page in pages:
            page.page_content = page.page_content.replace("\x00", " ").strip()
            page.metadata.update({"source": pdf_path.name, "source_path": str(pdf_path)})

        documents.extend(page for page in pages if page.page_content)

    if not documents:
        raise ValueError("The uploaded PDFs did not contain extractable text. Try OCR first for scanned PDFs.")

    return documents


langchain_docs = load_langchain_pdf_documents(pdf_paths)

langchain_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
langchain_split_docs = langchain_text_splitter.split_documents(langchain_docs)

print(f"Loaded {len(langchain_docs)} page document(s).")
print(f"Split them into {len(langchain_split_docs)} LangChain chunk(s).")
print()
print("First LangChain chunk preview:")
print(langchain_split_docs[0].page_content[:800])

In [ ]:
langchain_generator = TestsetGenerator(
    llm=langchain_generator_llm,
    embedding_model=langchain_generator_embeddings,
)

langchain_dataset = langchain_generator.generate_with_langchain_docs(
    langchain_split_docs,
    testset_size=TESTSET_SIZE,
)

### LangChain generated test set

In [ ]:
langchain_compact_df = compact_testset_dataframe(langchain_dataset)
langchain_compact_df

## 5. LlamaIndex test data generation

Load the same uploaded PDFs as LlamaIndex documents, split them with LlamaIndex, and generate another RAGAS test set.

In [ ]:
llamaindex_docs = SimpleDirectoryReader(
    input_files=[str(pdf_path) for pdf_path in pdf_paths]
).load_data()

llamaindex_splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
llamaindex_nodes = llamaindex_splitter.get_nodes_from_documents(llamaindex_docs)
llamaindex_split_docs = [
    LlamaIndexDocument(
        text=node.get_content().replace("\x00", " ").strip(),
        metadata=dict(node.metadata),
    )
    for node in llamaindex_nodes
    if node.get_content().strip()
]

if not llamaindex_split_docs:
    raise ValueError("The uploaded PDFs did not contain extractable text for LlamaIndex. Try OCR first for scanned PDFs.")

print(f"Loaded {len(llamaindex_docs)} LlamaIndex document(s).")
print(f"Split them into {len(llamaindex_split_docs)} LlamaIndex chunk document(s).")
print()
print("First LlamaIndex chunk preview:")
print(llamaindex_split_docs[0].text[:800])

In [ ]:
llamaindex_generator = TestsetGenerator.from_llama_index(
    llm=llamaindex_llm,
    embedding_model=llamaindex_embeddings,
)

llamaindex_dataset = llamaindex_generator.generate_with_llamaindex_docs(
    llamaindex_split_docs,
    testset_size=TESTSET_SIZE,
)

### LlamaIndex generated test set

In [ ]:
llamaindex_compact_df = compact_testset_dataframe(llamaindex_dataset)
llamaindex_compact_df

## 6. Export generated datasets

RAGAS creates its standard test-set schema during generation. To keep only the needed columns, export the compact DataFrames.

In [ ]:
from IPython.display import display

langchain_compact_df.to_csv("ragas_langchain_generated_testset.csv", index=False)
llamaindex_compact_df.to_csv("ragas_llamaindex_generated_testset.csv", index=False)

print("Saved:")
print("- ragas_langchain_generated_testset.csv")
print("- ragas_llamaindex_generated_testset.csv")

print("\nLangChain generated dataset:")
display(langchain_compact_df)

print("LlamaIndex generated dataset:")
display(llamaindex_compact_df)